## 批量提取并导出有用曲线（含单位规范化）

- 输入目录：`data/all_well_las_vp`
- 输出目录：`data/vertical_well_las_vp_vs_rho_interpre_raw`
- 规则来源：`import.csv`
- 提取类别：井眼质量、伽马/泥质、纵波声波、横波声波、密度、孔隙度、渗透率、含水饱和度
- 单位处理：
  - 纵波声波/横波声波：若单位精确匹配 `US/FT` 或 `US/F`，则换算为 `us/m`
  - 密度：若单位精确匹配 `K/M3`，则换算为 `g/cm3`
  - 输出时将这三类目标单位规范到 `us/m` 或 `g/cm3`
- 额外统计：维护三类曲线单位集合并在最后打印。


In [ ]:
import sys
from collections import Counter
from pathlib import Path

import lasio
import pandas as pd

# 兼容从工作区根目录或 notebooks 目录启动内核。
project_root = Path.cwd().resolve()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from cup.petrel.export import export_logsets_to_las
from cup.petrel.load import extract_any_log_from_las

input_dir = project_root / "data" / "all_well_las_vp"
output_dir = project_root / "data" / "vertical_well_las_vp_vs_rho_interpre_raw"
import_csv_path = project_root / "data" / "import.csv"

selected_categories = [
    "井眼质量",
    "伽马/泥质",
    "纵波声波",
    "横波声波",
    "密度",
    "孔隙度",
    "渗透率",
    "含水饱和度",
]

USFT_TO_USM = 3.280839895013123
KM3_TO_GCM3 = 0.001
DEBUG = True

sonic_categories = {"纵波声波", "横波声波"}
tracked_categories = ["纵波声波", "横波声波", "密度"]
units_seen: dict[str, set[str]] = {k: set() for k in tracked_categories}


def parse_mnemonics(raw_value: object) -> list[str]:
    if raw_value is None or pd.isna(raw_value):  # type: ignore
        return []
    text = str(raw_value).strip()
    if not text or text == "-":
        return []

    out: list[str] = []
    for token in text.split(","):
        mnemonic = token.strip()
        if mnemonic and mnemonic != "-":
            out.append(mnemonic)
    return out


def normalize_log_unit_by_category(log, category: str):
    raw_unit = "" if log.unit is None else str(log.unit).strip()

    if category in units_seen and raw_unit:
        units_seen[category].add(raw_unit)

    # 精确匹配规则：仅在命中指定写法时执行换算。
    if category in sonic_categories and raw_unit in {"US/FT", "US/F"}:
        log.series = pd.Series(data=log.values * USFT_TO_USM, name=log.name, index=log.series.index)
        log.unit = "us/m"
    elif category == "密度" and raw_unit == "K/M3":
        log.series = pd.Series(data=log.values * KM3_TO_GCM3, name=log.name, index=log.series.index)
        log.unit = "g/cm3"

    # 输出单位规范化（仅针对这三类）。
    if category in sonic_categories and str(log.unit).strip() in {"US/M", "us/m"}:
        log.unit = "us/m"
    if category == "密度" and str(log.unit).strip() in {"G/C3", "g/cm3"}:
        log.unit = "g/cm3"

    return log


def resolve_las_path(file_token: str) -> Path | None:
    token = str(file_token).strip()
    if not token or token == "nan":
        return None

    candidates = [token]
    if not token.lower().endswith(".las"):
        candidates.append(f"{token}.las")

    for name in candidates:
        p = input_dir / name
        if p.exists():
            return p
    return None


import_df = pd.read_csv(import_csv_path)
if "文件名" not in import_df.columns:
    raise ValueError("import.csv 缺少 '文件名' 列。")

missing_cols = [c for c in selected_categories if c not in import_df.columns]
if missing_cols:
    raise ValueError(f"import.csv 缺少类别列: {missing_cols}")

logsets_for_export: dict[str, dict] = {}
skipped_wells_local: list[dict[str, str]] = []
skipped_curves_local: list[dict[str, str]] = []

if DEBUG:
    print("[DEBUG] input_dir =", input_dir)
    print("[DEBUG] import_csv_path =", import_csv_path)
    print("[DEBUG] input_dir exists =", input_dir.exists())
    print("[DEBUG] output_dir =", output_dir)
    print("[DEBUG] import.csv 行数 =", len(import_df))
    print("[DEBUG] 文件名样例 =", import_df["文件名"].head(5).tolist())

for _, row in import_df.iterrows():
    file_name = str(row["文件名"]).strip()
    if not file_name or file_name == "nan":
        continue

    # 对接 well_select_and_io：默认文件名字段包含 .las 后缀。
    las_path = resolve_las_path(file_name)
    if las_path is None:
        skipped_wells_local.append({"well": file_name, "reason": "未找到 LAS 文件（已尝试原名和补 .las）"})
        continue

    # 导出函数会自动追加 .las，这里统一用 stem 作为井名键，避免 .las.las。
    well_name = las_path.stem

    if DEBUG and len(logsets_for_export) < 3:
        print(f"[DEBUG] matched file: {file_name} -> {las_path.name}; export_well = {well_name}")

    las_file = lasio.read(las_path)
    extracted_logs = {}

    for category in selected_categories:
        for mnemonic in parse_mnemonics(row[category]):
            if mnemonic in extracted_logs:
                continue
            try:
                extracted_log = extract_any_log_from_las(las_file, mnemonic)
                extracted_log = normalize_log_unit_by_category(extracted_log, category)
                extracted_logs[mnemonic] = extracted_log
            except Exception as exc:
                skipped_curves_local.append(
                    {
                        "well": well_name,
                        "curve": mnemonic,
                        "reason": str(exc),
                    }
                )

    if not extracted_logs:
        skipped_wells_local.append({"well": well_name, "reason": "8类目标曲线均未成功提取"})
        continue

    logsets_for_export[well_name] = extracted_logs

export_summary = export_logsets_to_las(
    logsets=logsets_for_export,  # type: ignore
    output_dir=output_dir,
    curve_names=None,
    null_value=-999.25,
)

summary = {
    "planned_wells": int(import_df["文件名"].notna().sum()),
    "prepared_for_export": len(logsets_for_export),
    "exported_files": len(export_summary["exported_files"]),
    "local_skipped_wells": len(skipped_wells_local),
    "local_skipped_curves": len(skipped_curves_local),
    "export_skipped_wells": len(export_summary["skipped_wells"]),
    "export_skipped_curves": len(export_summary["skipped_curves"]),
}

print("纵波声波 units:", sorted(units_seen["纵波声波"]))
print("横波声波 units:", sorted(units_seen["横波声波"]))
print("密度 units:", sorted(units_seen["密度"]))

if DEBUG:
    print("[DEBUG] summary =", summary)
    if skipped_wells_local:
        print("[DEBUG] skipped_wells_local 示例:")
        print(pd.DataFrame(skipped_wells_local).head(20))
        reason_count = Counter(item["reason"] for item in skipped_wells_local)
        print("[DEBUG] skipped_wells_local 原因统计:", dict(reason_count))
    if skipped_curves_local:
        print("[DEBUG] skipped_curves_local 示例:")
        print(pd.DataFrame(skipped_curves_local).head(20))
    if summary["exported_files"] == 0:
        print("[DEBUG] 未导出文件。优先检查：输入目录、文件名列内容、曲线 mnemonic 是否存在。")

pd.DataFrame([summary])